# Feature Exploration & Leakage Checks

In [ ]:
import sys
sys.path.insert(0, '..')
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
sns.set_style('whitegrid')

df = pl.read_parquet('../data/processed/features.parquet')
print(f'Features shape: {df.shape}')
print(f'Columns: {len(df.columns)}')

In [ ]:
# Null percentage per feature
null_pct = df.null_count() / df.shape[0] * 100
null_pct_df = pl.DataFrame({'col': null_pct.columns, 'null_pct': null_pct.row(0)})
null_pct_df = null_pct_df.sort('null_pct', descending=True).head(20)
print('Top 20 columns by null %:')
print(null_pct_df)

In [ ]:
# Leakage check: week 1 rolling features should be NULL
week1 = df.filter(pl.col('week') == 1)
rolling_cols = [c for c in df.columns if '_avg_' in c or '_szn_' in c]
print(f'Rolling feature columns: {len(rolling_cols)}')
for col in rolling_cols[:10]:
    non_null = week1[col].is_not_null().mean()
    print(f'  {col}: {non_null:.1%} non-null in week 1')

In [ ]:
# Feature correlation with target
numeric_cols = [c for c in df.columns if df[c].dtype in (pl.Float64, pl.Float32, pl.Int8, pl.Int64)]
correlations = {}
target = df['target'].to_numpy().astype(float)
for col in numeric_cols:
    vals = df[col].to_numpy().astype(float)
    mask = ~np.isnan(vals)
    if mask.sum() > 100:
        correlations[col] = np.corrcoef(vals[mask], target[mask])[0, 1]

corr_df = pl.DataFrame({'feature': list(correlations.keys()), 'corr': list(correlations.values())})
corr_df = corr_df.sort(pl.col('corr').abs(), descending=True).head(20)
print('Top 20 features by |correlation| with target:')
print(corr_df)